# 🚀 vLLM GPU Server & Cloudflare Remote Tunnel

This notebook turns your Google Colab instance into a high-performance **vLLM OpenAI-Compatible API Server** powered by NVIDIA GPUs (T4, A100, L4).

### Features:
- **Model**: `Qwen/Qwen2.5-1.5B-Instruct` (or custom HuggingFace model)
- **PagedAttention & Prefix Caching**: Enabled for zero-waste KV-cache reuse
- **Cloudflare Quick Tunnel**: Secure, zero-setup public HTTPS endpoint to connect your local FastAPI Gateway and load-testing suite

## 1. Verify NVIDIA GPU Hardware
Make sure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Install Dependencies (with Colab PyTorch fix)

In [4]:
# Fix Colab CUDA/TorchAudio mismatch
!pip uninstall -y torchaudio

# Install latest vLLM
!pip install -q -U vllm

# Download and install Cloudflare Tunnel CLI
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.8 MB/s eta 0:00:0000:01
(Reading database ... 122583 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.8.3) over (2026.8.3) ...
Setting up cloudflared (2026.8.3) ...
Processing triggers for man-db (2.10.2-1) ...


## 3. Launch vLLM & Cloudflare Tunnel
Run this cell to start the server. It will display the live `https://xxxx.trycloudflare.com` URL to copy into your local `.env` file.

In [ ]:
import os
import re
import subprocess
import sys
import time

# --- Configuration ---
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
API_KEY = "dev-secret"
PORT = "8000"
MAX_MODEL_LEN = "4096"
GPU_MEMORY_UTILIZATION = "0.90"

print(f"🚀 Starting vLLM server with model: {MODEL_NAME}...")

# 1. Start vLLM process
vllm_cmd = [
    "vllm", "serve", MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", PORT,
    "--api-key", API_KEY,
    "--enable-prefix-caching",
    "--max-model-len", MAX_MODEL_LEN,
    "--gpu-memory-utilization", GPU_MEMORY_UTILIZATION,
]
vllm_proc = subprocess.Popen(vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

vllm_ready = False
# Wait for vLLM to finish loading weights and start up
for line in vllm_proc.stdout:
    print(line, end="")
    if "Application startup complete" in line or "Uvicorn running on" in line:
        vllm_ready = True
        print("\n✅ vLLM Engine is UP and ready!\n")
        break

if not vllm_ready and vllm_proc.poll() is not None:
    print(f"\n❌ vLLM failed to start (exit code {vllm_proc.returncode}). Please check the traceback above.")
    sys.exit(1)

# 2. Start Cloudflare Quick Tunnel
print("🌐 Spawning Cloudflare Quick Tunnel...")
tunnel_cmd = ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for line in tunnel_proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "=" * 65)
        print(f"🎉 PUBLIC TUNNEL READY: {tunnel_url}")
        print("\n📝 Copy this into your local .env file:")
        print(f"VLLM_BASE_URL={tunnel_url}/v1")
        print(f"VLLM_API_KEY={API_KEY}")
        print(f"VLLM_MODEL={MODEL_NAME}")
        print("=" * 65 + "\n")
        break

# Keep the cell alive and handle termination gracefully
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down vLLM and Cloudflare tunnel...")
    vllm_proc.terminate()
    tunnel_proc.terminate()
    print("Stopped.")

🚀 Starting vLLM server with model: Qwen/Qwen2.5-1.5B-Instruct...
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:333] 
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-1.5B-Instruct
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:333] 
(APIServer pid=4698) INFO 09-01 11:41:06 [api_utils.py:272] non-default args: {'model_tag': 'Qwen/Qwen2.5-1.5B-Instruct', 'host': '0.0.0.0', 'api_key': ['dev-secret'], 'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'enable_prefix_caching': True}
(APIServer pid=4698) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable hig

## 4. (Optional) In-Colab Zero-Network Latency Benchmark
Test raw GPU continuous batching throughput directly inside Colab (bypassing internet round-trip latency).

In [ ]:
!python3 -m vllm.entrypoints.openai.bench_serving \
    --backend vllm \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --endpoint /v1/chat/completions \
    --dataset-name random \
    --random-input-len 256 \
    --random-output-len 128 \
    --num-prompts 50 \
    --request-rate 10 \
    --host localhost \
    --port 8000 \
    --api-key dev-secret